# Retail Intelligence Pipeline: Idempotent Incremental Version

This version bootstraps a separate Delta Silver state from the existing `orders_v2` Parquet dataset, merges only unprocessed incremental CSV files by `order_id`, and records successful files in a Delta manifest. Existing `orders_v2` and Gold paths are not deleted. Run top to bottom once, review counts and SQL server configuration, then validate a second run reports zero pending files.

**Important:** Confirm `sql_server` in the credentials cell matches the exact logical server hostname in Azure Portal before running the SQL publishing cells.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from functools import reduce

storage_account = "retaildatalakest"

bronze_root = f"abfss://bronze@{storage_account}.dfs.core.windows.net"
incremental_root = f"{bronze_root}/incremental"
original_orders_path = f"{bronze_root}/orders.csv"

# Keep the existing Parquet Silver path as the bootstrap source.
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/orders_v2"

# New Delta state path and manifest. These do not overwrite the existing Parquet dataset.
silver_delta_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/orders_delta_v1"
manifest_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/processed_files_manifest"

gold_root = f"abfss://gold@{storage_account}.dfs.core.windows.net"

print("Original Bronze source:", original_orders_path)
print("Incremental Bronze folder:", incremental_root)
print("Existing Silver bootstrap:", silver_path)
print("Incremental Delta Silver:", silver_delta_path)
print("Processed-file manifest:", manifest_path)
print("Gold destination:", gold_root)

In [ ]:
# Discover incremental CSVs and read the manifest.
# File paths are recorded only after their rows have been merged successfully.
all_incremental_files = sorted([
    item.path for item in dbutils.fs.ls(incremental_root)
    if item.name.lower().endswith(".csv")
])

manifest_schema = StructType([
    StructField("file_path", StringType(), False),
    StructField("status", StringType(), False),
    StructField("processed_at", TimestampType(), True)
])

if DeltaTable.isDeltaTable(spark, manifest_path):
    processed_files_df = (
        spark.read.format("delta").load(manifest_path)
        .filter(F.col("status") == "SUCCESS")
        .select("file_path")
        .distinct()
    )
    processed_files = {r["file_path"] for r in processed_files_df.collect()}
else:
    (
        spark.createDataFrame([], manifest_schema)
        .write.format("delta").mode("overwrite").save(manifest_path)
    )
    processed_files = set()

pending_files = [p for p in all_incremental_files if p not in processed_files]
print(f"Incremental files discovered: {len(all_incremental_files)}")
print(f"Previously processed files: {len(processed_files)}")
print(f"New files to process: {len(pending_files)}")

def normalize_orders(df):
    # Normalize types consistently for original, historical Silver and new CSV data.
    df = (
        df.withColumn("order_date", F.to_timestamp("order_date"))
          .withColumn("status", F.lower(F.trim(F.col("status"))))
          .withColumn("quantity", F.expr("try_cast(quantity AS DOUBLE)"))
          .withColumn("unit_price", F.expr("try_cast(unit_price AS DOUBLE)"))
          .withColumn("delivery_days", F.expr("try_cast(delivery_days AS DOUBLE)"))
          .withColumn("promised_days", F.expr("try_cast(promised_days AS DOUBLE)"))
    )
    if "ingested_at" in df.columns:
        df = df.withColumn("ingested_at", F.to_timestamp("ingested_at"))
    else:
        df = df.withColumn("ingested_at", F.lit(None).cast("timestamp"))
    return df.withColumn("order_month", F.date_format("order_date", "yyyy-MM"))

# Bootstrap Delta once from the existing cleaned Silver dataset when available.
# If that dataset is absent, bootstrap from the original Bronze file.
if not DeltaTable.isDeltaTable(spark, silver_delta_path):
    try:
        bootstrap_df = spark.read.format("parquet").load(silver_path)
        print("Bootstrapping Delta Silver from existing Silver Parquet.")
    except Exception as exc:
        print("Existing Silver Parquet not available; bootstrapping from original Bronze.")
        bootstrap_df = (
            spark.read.option("header", True).option("inferSchema", True)
            .csv(original_orders_path)
        )

    bootstrap_df = normalize_orders(bootstrap_df)
    # Keep one record per non-null order_id before initial state creation.
    order_window = Window.partitionBy("order_id").orderBy(
        F.col("ingested_at").desc_nulls_last(),
        F.col("order_date").desc_nulls_last()
    )
    bootstrap_df = (
        bootstrap_df
        .withColumn("_rn", F.when(F.col("order_id").isNotNull(), F.row_number().over(order_window)).otherwise(F.lit(1)))
        .filter(F.col("order_id").isNull() | (F.col("_rn") == 1))
        .drop("_rn")
    )
    bootstrap_df.write.format("delta").mode("overwrite").save(silver_delta_path)
    print(f"Delta Silver bootstrapped with {bootstrap_df.count():,} rows.")

# Read only files not yet recorded as successful.
if pending_files:
    raw_batches = [
        spark.read.option("header", True).option("inferSchema", True).csv(path)
        for path in pending_files
    ]
    incoming_df = reduce(
        lambda left, right: left.unionByName(right, allowMissingColumns=True),
        raw_batches
    )
    incoming_df = normalize_orders(incoming_df)

    # Do not silently mark invalid IDs as processed; stop so the source can be fixed.
    null_ids = incoming_df.filter(F.col("order_id").isNull()).count()
    if null_ids:
        raise ValueError(
            f"{null_ids} incoming rows have null order_id. Files were not marked processed."
        )

    # Deduplicate within the incoming batch before MERGE.
    order_window = Window.partitionBy("order_id").orderBy(
        F.col("ingested_at").desc_nulls_last(),
        F.col("order_date").desc_nulls_last()
    )
    incoming_df = (
        incoming_df
        .withColumn("_rn", F.row_number().over(order_window))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    delta_target = DeltaTable.forPath(spark, silver_delta_path)
    # MERGE makes retries safe: already merged order_ids are updated, not appended.
    (
        delta_target.alias("t")
        .merge(incoming_df.alias("s"), "t.order_id = s.order_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    # Record file success only after the Delta MERGE succeeds.
    manifest_rows = [(path, "SUCCESS") for path in pending_files]
    manifest_updates = (
        spark.createDataFrame(manifest_rows, ["file_path", "status"])
        .withColumn("processed_at", F.current_timestamp())
    )
    manifest_delta = DeltaTable.forPath(spark, manifest_path)
    (
        manifest_delta.alias("t")
        .merge(manifest_updates.alias("s"), "t.file_path = s.file_path")
        .whenMatchedUpdate(set={
            "status": "s.status",
            "processed_at": "s.processed_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"Merged {len(pending_files)} new file(s) and updated the manifest.")
else:
    print("No new incremental files. Existing Delta Silver state will be reused.")

# All downstream cells consume this curated Delta state.
silver_df = spark.read.format("delta").load(silver_delta_path)
print(f"Current Delta Silver rows: {silver_df.count():,}")
display(silver_df.orderBy(F.col("order_date").desc_nulls_last()).limit(10))

In [ ]:
# Read the canonical Delta Silver state. Do not overwrite Silver on every run.
silver_df = spark.read.format("delta").load(silver_delta_path)

print(f"Silver rows: {silver_df.count():,}")
print("Silver schema:")
silver_df.printSchema()

In [ ]:
total_rows = silver_df.count()

null_order_ids = silver_df.filter(F.col("order_id").isNull()).count()

duplicate_order_ids = (
    silver_df.filter(F.col("order_id").isNotNull())
    .groupBy("order_id").count()
    .filter(F.col("count") > 1).count()
)

invalid_prices = silver_df.filter(
    F.col("unit_price").isNull() | (F.col("unit_price") < 0)
).count()

invalid_quantities = silver_df.filter(
    F.col("quantity").isNull() | (F.col("quantity") <= 0)
).count()

invalid_dates = silver_df.filter(F.col("order_date").isNull()).count()

print(f"Total rows: {total_rows:,}")
print(f"Null order IDs: {null_order_ids}")
print(f"Duplicate order IDs: {duplicate_order_ids}")
print(f"Invalid prices: {invalid_prices}")
print(f"Invalid quantities: {invalid_quantities}")
print(f"Invalid order dates: {invalid_dates}")

if null_order_ids or duplicate_order_ids or invalid_prices or invalid_quantities or invalid_dates:
    raise ValueError("Data-quality checks failed. Gold and Azure SQL publishing stopped.")
else:
    print("Data-quality checks passed.")

In [0]:
status = F.lower(F.trim(F.col("status")))
is_cancelled = status.isin("canceled", "cancelled")
is_active = ~is_cancelled
revenue_expr = F.col("quantity") * F.col("unit_price")

active_order_count = F.countDistinct(
    F.when(is_active, F.col("order_id"))
)

kpis = silver_df.agg(
    F.countDistinct("order_id").cast("long").alias("total_orders"),
    F.round(
        F.sum(F.when(is_active, revenue_expr).otherwise(F.lit(0.0))), 2
    ).cast("decimal(18,2)").alias("net_revenue"),
    F.round(
        F.sum(F.when(is_active, revenue_expr)) / active_order_count, 2
    ).cast("decimal(18,2)").alias("average_order_value"),
    F.round(
        F.avg(
            F.when(
                is_active,
                F.when(F.col("delivery_days") > F.col("promised_days"), 1.0)
                 .otherwise(0.0)
            )
        ) * 100, 2
    ).cast("decimal(6,2)").alias("late_delivery_pct"),
    F.round(
        F.avg(F.when(is_cancelled, 1.0).otherwise(0.0)) * 100, 2
    ).cast("decimal(6,2)").alias("cancellation_pct")
).withColumn("updated_at", F.current_timestamp())

daily_sales = (
    silver_df
    .filter(is_active)
    .withColumn("order_day", F.to_date("order_date"))
    .withColumn("line_revenue", revenue_expr)
    .groupBy("order_day")
    .agg(
        F.countDistinct("order_id").cast("long").alias("total_orders"),
        F.round(F.sum("line_revenue"), 2)
         .cast("decimal(18,2)").alias("revenue")
    )
    .orderBy("order_day")
)

product_performance = (
    silver_df
    .filter(is_active)
    .withColumn("line_revenue", revenue_expr)
    .groupBy("category")
    .agg(
        F.countDistinct("order_id").cast("long").alias("total_orders"),
        F.sum(F.col("quantity").cast("long")).cast("long").alias("units_sold"),
        F.round(F.sum("line_revenue"), 2)
         .cast("decimal(18,2)").alias("revenue")
    )
    .orderBy(F.desc("revenue"))
)

print("Gold KPIs:")
display(kpis)
print("Gold daily sales:")
display(daily_sales.limit(10))
print("Gold product performance:")
display(product_performance)

Gold KPIs:


total_orders,net_revenue,average_order_value,late_delivery_pct,cancellation_pct,updated_at
20200,20436382.11,1269.58,31.43,20.31,2026-10-25T05:41:59.895Z


Gold daily sales:


order_day,total_orders,revenue
2025-01-01,52,67280.18
2025-01-02,50,60043.88
2025-01-03,47,56363.02
2025-01-04,37,56046.19
2025-01-05,42,56521.65
2025-01-06,46,66374.44
2025-01-07,46,55764.07
2025-01-08,42,44515.41
2025-01-09,50,64794.10
2025-01-10,56,70629.83


Gold product performance:


category,total_orders,units_sold,revenue
Sports,2751,8154,3546717.20
Beauty,2651,7896,3412600.80
Grocery,2666,7975,3403351.27
Fashion,2627,7889,3378542.89
Electronics,2683,7955,3369769.51
Home,2692,7905,3306055.36
Clothing,27,86,19345.08


In [0]:
gold_datasets = [
    ("kpis", kpis),
    ("daily_sales", daily_sales),
    ("product_performance", product_performance),
]

for dataset_name, dataframe in gold_datasets:
    (
        dataframe.write
        .format("parquet")
        .mode("overwrite")
        .parquet(f"{gold_root}/{dataset_name}")
    )
    print(f"Wrote Gold dataset: {dataset_name}")

print("All Gold Parquet datasets written.")

Wrote Gold dataset: kpis
Wrote Gold dataset: daily_sales
Wrote Gold dataset: product_performance
All Gold Parquet datasets written.


In [0]:
for dataset_name, dataframe in gold_datasets:
    print(f"\n{dataset_name}: {dataframe.count():,} rows")
    dataframe.printSchema()


kpis: 1 rows
root
 |-- total_orders: long (nullable = false)
 |-- net_revenue: decimal(18,2) (nullable = true)
 |-- average_order_value: decimal(18,2) (nullable = true)
 |-- late_delivery_pct: decimal(6,2) (nullable = true)
 |-- cancellation_pct: decimal(6,2) (nullable = true)
 |-- updated_at: timestamp (nullable = false)


daily_sales: 368 rows
root
 |-- order_day: date (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- revenue: decimal(18,2) (nullable = true)


product_performance: 7 rows
root
 |-- category: string (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- units_sold: long (nullable = true)
 |-- revenue: decimal(18,2) (nullable = true)



In [0]:
sql_server = "REPLACE_WITH_EXACT_AZURE_SQL_SERVER_HOSTNAME"
sql_database = "retail-analytics-db"

# Credentials are retrieved from Unity Catalog secrets, not hard-coded here.
sql_user = dbutils.secrets.get(
    catalog="adb-dp750",
    schema="default",
    key="azure-sql-username"
)

sql_password = dbutils.secrets.get(
    catalog="adb-dp750",
    schema="default",
    key="azure-sql-password"
)

print("Azure SQL configuration loaded for:", sql_database)

Azure SQL configuration loaded for: retail-analytics-db


In [0]:
sql_test = (
    spark.read.format("sqlserver")
    .option("host", sql_server)
    .option("port", "1433")
    .option("database", sql_database)
    .option("query", """
        SELECT 'gold_kpis' AS table_name, COUNT(*) AS row_count
        FROM dbo.gold_kpis
        UNION ALL
        SELECT 'gold_daily_sales', COUNT(*)
        FROM dbo.gold_daily_sales
        UNION ALL
        SELECT 'gold_product_performance', COUNT(*)
        FROM dbo.gold_product_performance
    """)
    .option("user", sql_user)
    .option("password", sql_password)
    .load()
)

display(sql_test)

table_name,row_count
gold_kpis,1
gold_daily_sales,368
gold_product_performance,7


In [0]:
def publish_to_azure_sql(dataframe, table_name):
    (
        dataframe.write
        .format("sqlserver")
        .option("host", sql_server)
        .option("port", "1433")
        .option("database", sql_database)
        .option("dbtable", f"dbo.{table_name}")
        .option("user", sql_user)
        .option("password", sql_password)
        .option("truncate", "true")
        .mode("overwrite")
        .save()
    )
    print(f"Published dbo.{table_name}")

# Match the SQL table's expected column names and numeric types.
kpis_to_sql = (
    kpis
    .withColumn("id", F.lit(1).cast("int"))
    .select(
        "id",
        "total_orders",
        "net_revenue",
        "average_order_value",
        "late_delivery_pct",
        "cancellation_pct",
        "updated_at"
    )
)

daily_sales_to_sql = daily_sales.select(
    F.col("order_day").cast("date").alias("order_day"),
    F.col("total_orders").cast("long").alias("total_orders"),
    F.col("revenue").cast("decimal(18,2)").alias("revenue")
)

product_performance_to_sql = product_performance.select(
    F.col("category").cast("string").alias("category"),
    F.col("total_orders").cast("long").alias("total_orders"),
    F.col("units_sold").cast("long").alias("units_sold"),
    F.col("revenue").cast("decimal(18,2)").alias("revenue")
)

publish_to_azure_sql(kpis_to_sql, "gold_kpis")
publish_to_azure_sql(daily_sales_to_sql, "gold_daily_sales")
publish_to_azure_sql(product_performance_to_sql, "gold_product_performance")

print("SUCCESS: all Gold datasets published to Azure SQL.")

Published dbo.gold_kpis
Published dbo.gold_daily_sales
Published dbo.gold_product_performance
SUCCESS: all Gold datasets published to Azure SQL.
